In [2]:
import os
import sys
import glob
import ccdproc
import logging
import warnings
import numpy as np
from astropy.io import fits
import ipywidgets as widgets
from ccdproc import Combiner
import astropy.io.fits as fits
from astropy import units as u
import matplotlib.pyplot as plt
from ccdproc import wcs_project
from astropy.nddata import NDData
from astropy.nddata import CCDData
from IPython.display import display
from astropy.io.fits import getheader
from scipy.ndimage import uniform_filter
from astropy.nddata import fits_ccddata_writer
from astropy.nddata import fits_ccddata_reader
from astropy.utils.misc import NumpyRNGContext
from ccdproc import Combiner, fits_ccddata_reader
from astropy.modeling.functional_models import Gaussian2D

warnings.filterwarnings('ignore')
sys.stderr = open(os.devnull, "w")
warnings.filterwarnings('ignore', category=UserWarning)

In [4]:
carpeta = "C:/Users/HP/Desktop/Maestria/TESIS/R20241009_66146"
archivos_fits = glob.glob(f"{carpeta}/*o.fits")
data = [fits_ccddata_reader(archivo, hdu=0, unit="adu") for archivo in archivos_fits]
print(f"Se han leído {len(data)} archivos FITS.")


Se han leído 99 archivos FITS.


In [6]:
procesadas = []

for imagen in data:
    data_with_deviation = ccdproc.create_deviation(
        imagen, gain=1.5 * u.electron/u.adu,
        readnoise=5 * u.electron
    )
    data_with_deviation.header['EXPTIME'] = 2.0 
    gain_corrected = ccdproc.gain_correct(data_with_deviation, 1.5 * u.electron/u.adu)
    cr_cleaned = ccdproc.cosmicray_lacosmic(gain_corrected, sigclip=5)
    procesadas.append(cr_cleaned)

print(f"Se procesaron {len(procesadas)} imágenes.")


Se procesaron 99 imágenes.


In [7]:
#-------------------------
# Creando el Master Bias
#-------------------------

archivos_bias = glob.glob(f"{carpeta}/*b.fits")
bias_list = [fits_ccddata_reader(archivo, hdu=0, unit="adu") for archivo in archivos_bias]
bias = Combiner(bias_list)
bias_master = bias.median_combine()
bias_master.write(os.path.join(carpeta, "Master_bias.fits"), overwrite=True)
print(f"Se combinaron {len(bias_list)} archivos BIAS en 'Master_bias.fits'")

#--------------------
# Correción por Bias
#--------------------
master_bias = fits_ccddata_reader(os.path.join(carpeta, "Master_bias.fits"), hdu=0, unit="adu") 
master_bias.header['EXPTIME'] = 0.0
data_bias_subtracted = [ccdproc.subtract_bias(imagen, master_bias) for imagen in data]
print(f"Se han corregido {len(data_bias_subtracted)} imágenes con el Master bias.")


Se combinaron 20 archivos BIAS en 'Master_bias.fits'


In [14]:

archivos_flat = glob.glob(f"{carpeta}/*f.fits")

In [16]:
# Lee los FLAT y los almacena en una lista
flat_list = [fits_ccddata_reader(archivo, hdu=0, unit="adu") for archivo in archivos_flat]

# Combinar todos los FLAT de la lista
flat = Combiner(flat_list)
flat_master = flat.median_combine()

# Guarda el FLAT combinado como un nuevo archivo FITS
flat_master.write(os.path.join(carpeta, "Master_flat.fits"), overwrite=True)

print(f"Se combinaron {len(flat_list)} archivos Flat en 'Master_flat.fits'")


Se combinaron 15 archivos Flat en 'Master_flat.fits'


In [18]:
master_flat = fits_ccddata_reader(os.path.join(carpeta, "Master_flat.fits"), hdu=0, unit="adu") 
master_flat.header['EXPTIME'] = 0.25

# 3. Aplicar la corrección con el master flat
data_reduced = [ccdproc.flat_correct(imagen, master_flat, min_value=0.9, norm_value=30000) for imagen in data_bias_subtracted]

print(f"Se han procesado {len(data_reduced)} imágenes con bias y flat.")


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
Se han procesado 99 imágenes con bias y flat.


In [25]:

# 1. Ignorar cualquier advertencia (Warning) de Python/Astropy
warnings.filterwarnings('ignore')
# 2. Silenciar los mensajes de registro (Logs)
# Esto evita que salgan los mensajes de "Negative values..."
logging.getLogger().setLevel(logging.ERROR)

# -----------------------------------------------------------------------------
# 0. CONFIGURACIÓN
# -----------------------------------------------------------------------------
carpeta = 'C:/Users/HP/Desktop/Maestria/TESIS/R20241009_66146'  # <--- CAMBIA ESTO POR LA RUTA DE TU CARPETA
print(f"Trabajando en la carpeta: {carpeta}")

# -----------------------------------------------------------------------------
# 1. CREACIÓN DEL MASTER BIAS
# -----------------------------------------------------------------------------
print("\n--- 1. Generando Master Bias ---")
archivos_bias = glob.glob(os.path.join(carpeta, "*b.fits"))

if archivos_bias:
    # Lee los BIAS
    bias_list = [fits_ccddata_reader(archivo, hdu=0, unit="adu") for archivo in archivos_bias]
    
    # Combinar
    bias_combiner = Combiner(bias_list)
    # Recomendación: Usar sigma clipping para eliminar valores extraños antes de la mediana
    bias_combiner.sigma_clipping(low_thresh=2, high_thresh=5, func=np.ma.median)
    master_bias = bias_combiner.median_combine()
    
    # Guardar
    master_bias.write(os.path.join(carpeta, "Master_bias.fits"), overwrite=True)
    print(f"-> Se creó 'Master_bias.fits' a partir de {len(bias_list)} archivos.")
else:
    print("-> No se encontraron archivos *b.fits. Intentando cargar Master_bias existente...")
    try:
        master_bias = fits_ccddata_reader(os.path.join(carpeta, "Biasmaster.fits"), hdu=0, unit="adu")
    except:
        raise FileNotFoundError("No se encontró ni archivos de bias ni un Master_bias.fits")

# Ajuste de cabecera necesario para ccdproc en algunos casos
master_bias.header['EXPTIME'] = 0.0

# -----------------------------------------------------------------------------
# 2. CREACIÓN DEL MASTER FLAT
# -----------------------------------------------------------------------------
print("\n--- 2. Generando Master Flat ---")
archivos_flat = glob.glob(os.path.join(carpeta, "*f.fits"))

if archivos_flat:
    flat_list = []
    for archivo in archivos_flat:
        # Leemos el flat crudo
        flat_raw = fits_ccddata_reader(archivo, hdu=0, unit="adu")
        # IMPORTANTE: Hay que restar el Bias a cada Flat individual antes de combinarlos
        flat_bias_sub = ccdproc.subtract_bias(flat_raw, master_bias)
        flat_list.append(flat_bias_sub)

    # Combinar
    flat_combiner = Combiner(flat_list)
    flat_combiner.sigma_clipping(low_thresh=2, high_thresh=5, func=np.ma.median)
    master_flat = flat_combiner.median_combine()
    
    # Guardar
    master_flat.write(os.path.join(carpeta, "Master_flat.fits"), overwrite=True)
    print(f"-> Se creó 'Master_flat.fits' a partir de {len(flat_list)} archivos.")
else:
    print("-> No se encontraron archivos *f.fits. Intentando cargar Master_flat existente...")
    try:
        master_flat = fits_ccddata_reader(os.path.join(carpeta, "Flatmaster.fits"), hdu=0, unit="adu")
    except:
        raise FileNotFoundError("No se encontró ni archivos flat ni un Master_flat.fits")

master_flat.header['EXPTIME'] = 0.25 # Valor de ejemplo que tenías en tu código

# -----------------------------------------------------------------------------
# 3. PROCESAMIENTO DE IMÁGENES CIENTÍFICAS (*o.fits -> *o2.fits)
# -----------------------------------------------------------------------------
print("\n--- 3. Procesando Imágenes Científicas ---")
archivos_ciencia = glob.glob(os.path.join(carpeta, "*o.fits")) # Busca los *o.fits

count = 0

for archivo_nombre in archivos_ciencia:
    print(f"Procesando: {os.path.basename(archivo_nombre)}...")
    
    # A. Cargar imagen
    raw_image = fits_ccddata_reader(archivo_nombre, hdu=0, unit="adu")
    
    # B. Restar Bias
    image_bias_sub = ccdproc.subtract_bias(raw_image, master_bias)
    
    # C. Corregir Flat
    # (min_value y norm_value dependen de tu cámara, los dejé como los tenías)
    image_flat_corr = ccdproc.flat_correct(image_bias_sub, master_flat, min_value=0.9, norm_value=30000)
    
    # D. Generar mapa de desviación (Ruido) para LA Cosmic
    # Nota: gain y readnoise deben coincidir con las specs de tu cámara CCD
    image_with_dev = ccdproc.create_deviation(
        image_flat_corr, 
        gain=1.5 * u.electron/u.adu, 
        readnoise=5 * u.electron
    )
    
    # Modificación manual de EXPTIME 
    image_with_dev.header['EXPTIME'] = 2.0 

    # E. Corrección de Ganancia (pasar de ADU a electrones)
    gain_corrected = ccdproc.gain_correct(image_with_dev, 1.5 * u.electron/u.adu)
    
    # F. Limpieza de Rayos Cósmicos (L.A. Cosmic)
    # sigclip=5 es estándar, objlim aumenta si detecta estrellas como ruido
    cr_cleaned = ccdproc.cosmicray_lacosmic(gain_corrected, sigclip=5)
    
    # -------------------------------------------------------------------------
    # GUARDAR EL RESULTADO
    # -------------------------------------------------------------------------
    # Creamos el nuevo nombre: ejemplo.o.fits -> ejemplo.o2.fits
    nombre_salida = archivo_nombre.replace('o.fits', 'o2.fits')
    
    # Guardamos (overwrite=True sobrescribe si ya existe)
    cr_cleaned.write(nombre_salida, overwrite=True)
    
    count += 1

print(f"\n¡Listo! Se procesaron y guardaron {count} imágenes con terminación *o2.fits")

Trabajando en la carpeta: C:/Users/HP/Desktop/Maestria/TESIS/R20241009_66146

--- 1. Generando Master Bias ---
-> Se creó 'Master_bias.fits' a partir de 20 archivos.

--- 2. Generando Master Flat ---
-> Se creó 'Master_flat.fits' a partir de 15 archivos.

--- 3. Procesando Imágenes Científicas ---
Procesando: 66146_20s_R_b2_0001o.fits...
Procesando: 66146_20s_R_b2_0002o.fits...
Procesando: 66146_20s_R_b2_0003o.fits...
Procesando: 66146_20s_R_b2_0004o.fits...
Procesando: 66146_20s_R_b2_0005o.fits...
Procesando: 66146_20s_R_b2_0006o.fits...
Procesando: 66146_20s_R_b2_0007o.fits...
Procesando: 66146_20s_R_b2_0008o.fits...
Procesando: 66146_20s_R_b2_0009o.fits...
Procesando: 66146_20s_R_b2_0010o.fits...
Procesando: 66146_20s_R_b2_0011o.fits...
Procesando: 66146_20s_R_b2_0012o.fits...
Procesando: 66146_20s_R_b2_0013o.fits...
Procesando: 66146_20s_R_b2_0014o.fits...
Procesando: 66146_20s_R_b2_0015o.fits...
Procesando: 66146_20s_R_b2_0016o.fits...
Procesando: 66146_20s_R_b2_0017o.fits...
Proc

In [ ]:
#%matplotlib widget
# Crear la figura y los ejes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Mostrar las imágenes (extrae los datos con .data)
im1 = ax1.imshow(data_reduced[0].data, origin='lower', cmap='gray')
ax1.set_title('Imagen_limpia')
plt.colorbar(im1, ax=ax1, label='Intensidad (ADU)')

im2 = ax2.imshow(data[0].data, origin='lower', cmap='gray')
ax2.set_title('Imagen_tomada')
plt.colorbar(im2, ax=ax2, label='Intensidad (ADU)')

plt.tight_layout()

# Extraer los valores numéricos mínimos y máximos
vmin_value = float(data[0].data.min())
vmax_value = float(data[0].data.max())

# Crear sliders para ajustar la escala de grises
vmin_slider = widgets.FloatSlider(value=vmin_value, min=vmin_value, max=vmax_value, step=10, description='vmin:')
vmax_slider = widgets.FloatSlider(value=vmax_value, min=vmin_value, max=vmax_value, step=10, description='vmax:')

# Función para actualizar la escala de grises
def update_image(vmin, vmax):
    im1.set_clim(vmin, vmax)
    im2.set_clim(vmin, vmax)
    fig.canvas.draw_idle()

# Conectar los sliders a la función de actualización
widgets.interactive(update_image, vmin=vmin_slider, vmax=vmax_slider)

# Mostrar los sliders
display(vmin_slider, vmax_slider)


In [ ]:
#%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import widgets, interact, Layout
from matplotlib.colors import LogNorm, PowerNorm

class FITSViewer:
    def __init__(self, data_clean, data_original):
        self.data_clean = data_clean
        self.data_original = data_original
        self.current_norm = 'linear'
        self.fig, (self.ax1, self.ax2) = plt.subplots(1, 2, figsize=(12, 6))
        
        # Configuración inicial
        self.vmin, self.vmax = self.get_zscale_range()
        self.gamma = 1.0
        
        # Crear imágenes
        self.im1 = self.ax1.imshow(self.data_clean, origin='lower', 
                                  cmap='gray', vmin=self.vmin, vmax=self.vmax)
        self.ax1.set_title('Imagen Limpia')
        self.cbar1 = self.fig.colorbar(self.im1, ax=self.ax1)
        
        self.im2 = self.ax2.imshow(self.data_original, origin='lower', 
                                  cmap='gray', vmin=self.vmin, vmax=self.vmax)
        self.ax2.set_title('Imagen Original')
        self.cbar2 = self.fig.colorbar(self.im2, ax=self.ax2)
        
        plt.tight_layout()
        
        # Crear controles
        self.create_controls()
        
    def get_zscale_range(self):
        """Calcula el rango zscale similar a SAOImage"""
        data = np.concatenate([self.data_clean.flatten(), self.data_original.flatten()])
        data = data[np.isfinite(data)]
        sorted_data = np.sort(data)
        n = len(sorted_data)
        
        # Algoritmo similar a zscale
        if n > 0:
            z1 = sorted_data[int(0.1*n)]
            z2 = sorted_data[int(0.9*n)]
            return z1, z2
        return np.min(data), np.max(data)
    
    def update_display(self):
        """Actualiza la visualización según los parámetros actuales"""
        norm_map = {
            'linear': None,
            'log': LogNorm(vmin=max(1e-10, self.vmin), vmax=self.vmax),
            'sqrt': PowerNorm(gamma=0.5, vmin=self.vmin, vmax=self.vmax),
            'power': PowerNorm(gamma=self.gamma, vmin=self.vmin, vmax=self.vmax)
        }
        
        norm = norm_map[self.current_norm]
        
        self.im1.set_norm(norm)
        self.im2.set_norm(norm)
        self.im1.set_clim(self.vmin, self.vmax)
        self.im2.set_clim(self.vmin, self.vmax)
        
        self.cbar1.update_normal(self.im1)
        self.cbar2.update_normal(self.im2)
        
        self.fig.canvas.draw_idle()
    
    def create_controls(self):
        """Crea los widgets de control interactivo"""
        # Selector de escala
        scale_selector = widgets.RadioButtons(
            options=['linear', 'log', 'sqrt', 'power'],
            value='linear',
            description='Escala:',
            disabled=False
        )
        
        # Sliders para rango
        data_range = self.get_zscale_range()
        full_min, full_max = np.nanmin([self.data_clean, self.data_original]), np.nanmax([self.data_clean, self.data_original])
        
        vmin_slider = widgets.FloatSlider(
            value=data_range[0],
            min=full_min,
            max=full_max,
            step=(full_max-full_min)/1000,
            description='Min:',
            continuous_update=False,
            layout=Layout(width='500px')
        )
        
        vmax_slider = widgets.FloatSlider(
            value=data_range[1],
            min=full_min,
            max=full_max,
            step=(full_max-full_min)/1000,
            description='Max:',
            continuous_update=False,
            layout=Layout(width='500px')
        )
        
        # Slider para gamma (solo visible en modo power)
        gamma_slider = widgets.FloatSlider(
            value=1.0,
            min=0.1,
            max=3.0,
            step=0.1,
            description='Gamma:',
            continuous_update=False,
            layout=Layout(width='500px'),
            style={'description_width': 'initial'}
        )
        
        # Botón para resetear a zscale
        reset_button = widgets.Button(
            description='Reset (ZScale)',
            layout=Layout(width='150px')
        )
        
        # Callbacks
        def on_scale_change(change):
            self.current_norm = change['new']
            self.update_display()
            gamma_slider.layout.visibility = 'visible' if self.current_norm == 'power' else 'hidden'
        
        def on_slider_change(change):
            self.vmin = vmin_slider.value
            self.vmax = vmax_slider.value
            self.update_display()
        
        def on_gamma_change(change):
            self.gamma = change['new']
            self.update_display()
        
        def on_reset_click(b):
            self.vmin, self.vmax = self.get_zscale_range()
            vmin_slider.value = self.vmin
            vmax_slider.value = self.vmax
            self.update_display()
        
        # Asignar callbacks
        scale_selector.observe(on_scale_change, names='value')
        vmin_slider.observe(on_slider_change, names='value')
        vmax_slider.observe(on_slider_change, names='value')
        gamma_slider.observe(on_gamma_change, names='value')
        reset_button.on_click(on_reset_click)
        
        # Organizar controles
        controls = widgets.VBox([
            widgets.HBox([scale_selector, reset_button]),
            vmin_slider,
            vmax_slider,
            gamma_slider
        ])
        
        display(controls)
        self.update_display()

# Uso:
viewer = FITSViewer(data_reduced[0].data, data[0].data)